# Урок 31 — Web Scraping: Читаємо українські новини

## requests · BeautifulSoup · pandas · Text Mining · WordCloud

**Модуль 4 · Network & Concurrent Systems · Автор: Viktor Nikoriak**

---

# Про що цей урок?

У попередньому уроці ми вивчили сокети та HTTP на низькому рівні.

Тепер ми піднімаємося на рівень вище — та навчимося **автоматично читати веб-сторінки**.

Більшість сайтів у інтернеті **не мають API**. Вони повертають **HTML** — текст зі спеціальною розміткою,
яку браузер перетворює у сторінку.

**Web scraping** — це мистецтво автоматизованого читання HTML.

Сьогодні ми побудуємо повний Data Science pipeline:

```
requests.get("https://www.rbc.ua/")
       ↓  HTTP GET
HTML response (текст сторінки)
       ↓
BeautifulSoup — парсимо HTML
       ↓
Витягуємо новини RBC Ukraine
       ↓
pandas DataFrame
       ↓
Text Mining (токени, стоп-слова, частоти)
       ↓
Word Frequency Analysis
       ↓
WordCloud + matplotlib
```


# Learning Objectives

Після цього уроку ти зможеш:

## Architecture
- Пояснити різницю між **API (JSON)** та **HTML scraping**
- Описати структуру HTML-документу: теги, атрибути, DOM-дерево
- Читати `robots.txt` та розуміти правила ethical scraping

## Practical Skills
- Виконувати `requests.get()` з правильними headers та timeout
- Парсити HTML через `find()`, `find_all()`, `select()`, `select_one()`
- Будувати `list[dict]` зі scraped даних та конвертувати у `pandas DataFrame`
- Очищати **українськомовний** текст: токенізація, стоп-слова, частоти
- Генерувати **WordCloud з підтримкою кирилиці**
- Будувати `matplotlib` charts для візуалізації новин

## Mental Model

```
HTML tag       = контейнер
CSS class      = тип контейнера
BeautifulSoup  = розумний читач HTML
find()         = знайди ПЕРШИЙ такий контейнер
find_all()     = знайди ВСІ такі контейнери
.text          = дістань текст зсередини
["href"]       = дістань значення атрибуту
.get("href")   = безпечно (не кидає KeyError)
```


# 1. Архітектура: Web Scraping Pipeline

```
┌─────────────────────────────────────────────────────────────────┐
│  КРОК 1: HTTP GET REQUEST                                       │
│                                                                  │
│  requests.get("https://www.rbc.ua/", headers=..., timeout=10)   │
│  → response.status_code  # 200 = OK                             │
│  → response.headers      # Content-Type, Server...              │
│  → response.text         # HTML як рядок                        │
└──────────────────────────────┬──────────────────────────────────┘
                               ↓
┌─────────────────────────────────────────────────────────────────┐
│  КРОК 2: HTML PARSING                                           │
│                                                                  │
│  soup = BeautifulSoup(html, "html.parser")                      │
│  → DOM-дерево Python-об'єктів                                   │
│  → soup.find_all("div", class_="newsline__item")                │
└──────────────────────────────┬──────────────────────────────────┘
                               ↓
┌─────────────────────────────────────────────────────────────────┐
│  КРОК 3: EXTRACT + STRUCTURE                                    │
│                                                                  │
│  news = []                                                       │
│  for item in soup.find_all(...):                                 │
│      news.append({"title": ..., "url": ..., "category": ...})   │
└──────────────────────────────┬──────────────────────────────────┘
                               ↓
┌─────────────────────────────────────────────────────────────────┐
│  КРОК 4: PANDAS + TEXT MINING                                   │
│                                                                  │
│  df = pd.DataFrame(news)                                         │
│  tokens = tokenize(df["title"].str.cat())                       │
│  freq = Counter(tokens).most_common(20)                          │
└──────────────────────────────┬──────────────────────────────────┘
                               ↓
┌─────────────────────────────────────────────────────────────────┐
│  КРОК 5: VISUALIZATION                                          │
│                                                                  │
│  WordCloud(freq).generate()                                      │
│  plt.barh(words, counts)                                        │
└─────────────────────────────────────────────────────────────────┘
```

## Що таке HTML?

HTML (HyperText Markup Language) — мова розмітки. Це **текст з тегами**.

```html
<div class="newsline__item">
  <a class="newsline__item-title" href="/ukr/news/123">
    Заголовок новини
  </a>
  <span class="newsline__item-category">Економіка</span>
  <time class="newsline__item-date">2024-05-08</time>
</div>
```

Браузер читає цей текст і **малює сторінку**.
Python через `BeautifulSoup` читає цей текст і **дістає дані**.


In [ ]:
# Встановлення необхідних бібліотек
# (виконай цю клітинку першою якщо деяких бібліотек ще немає)
!pip install requests beautifulsoup4 pandas matplotlib wordcloud --quiet

In [ ]:
# === ІМПОРТИ ===
import requests                          # HTTP-запити
from bs4 import BeautifulSoup           # HTML-парсер
import pandas as pd                      # таблиці та аналіз
import matplotlib.pyplot as plt          # графіки
import matplotlib
from collections import Counter          # підрахунок частот
import re                                # регулярні вирази
import time                              # затримки між запитами
import warnings
warnings.filterwarnings("ignore")

# Налаштування matplotlib для українських символів
matplotlib.rcParams["figure.figsize"] = (12, 6)
matplotlib.rcParams["font.size"] = 11

print("Всі бібліотеки імпортовано успішно")
print(f"   requests: {requests.__version__}")
print(f"   pandas:   {pd.__version__}")

# 2. Ethical Scraping: robots.txt та правила

**Перед тим як скрапити будь-який сайт — перевір `/robots.txt`.**

`robots.txt` — це файл на сервері, який повідомляє ботам які частини сайту можна сканувати:

```
# https://www.rbc.ua/robots.txt
User-agent: *
Disallow: /admin/
Disallow: /private/
Allow: /
Crawl-delay: 5
```

## Правила ввічливого скрапера

| Роби | НЕ роби |
|------|--------|
| Вказуй `User-Agent` | Приховувати хто ти |
| `time.sleep()` між запитами | Паралельний scraping без обмежень |
| Поважай `robots.txt` | Ігнорувати `Disallow` |
| Використовуй API якщо є | Scraping там де є офіційний API |
| Читай Terms of Service | Ігнорувати ToS |

## Наш User-Agent

```python
headers = {
    "User-Agent": "Mozilla/5.0 (Educational Demo — Python Course)"
}
```

Він чесно повідомляє сайту що це **навчальний демо-скрапер**.


In [ ]:
# === ПЕРЕВІРКА robots.txt ===
import urllib.robotparser

def check_robots(site_url: str, path: str = "/") -> bool:
    """Перевіряє чи дозволено scraping для заданого шляху."""
    rp = urllib.robotparser.RobotFileParser()
    robots_url = site_url.rstrip("/") + "/robots.txt"
    rp.set_url(robots_url)
    try:
        rp.read()
        allowed = rp.can_fetch("*", site_url + path)
        crawl_delay = rp.crawl_delay("*") or 1.0
        print(f"robots.txt: {robots_url}")
        print(f"Шлях {path!r}: {'ДОЗВОЛЕНО' if allowed else 'ЗАБОРОНЕНО'}")
        print(f"Рекомендована затримка між запитами: {crawl_delay}s")
        return allowed
    except Exception as e:
        print(f"Не вдалося завантажити robots.txt: {e}")
        return True  # відсутній robots.txt — зазвичай означає дозволено

print("=== Перевірка rbc.ua ===")
check_robots("https://www.rbc.ua", "/")

# 3. HTTP GET Request: отримуємо HTML

`requests.get()` робить те саме що браузер — надсилає **HTTP GET запит**.

Різниця: браузер **відображає** HTML графічно, Python отримує його як **рядок**.

## Що означає кожен параметр?

```python
requests.get(
    url,              # адреса ресурсу
    headers={...},    # HTTP заголовки (User-Agent, Accept-Language...)
    timeout=10,       # максимум 10 секунд на відповідь
)
```

## HTTP Status Codes

| Код | Значення |
|-----|----------|
| 200 | OK — все добре |
| 301/302 | Redirect — сторінка переїхала |
| 403 | Forbidden — доступ заборонено |
| 404 | Not Found — сторінка не знайдена |
| 429 | Too Many Requests — занадто часто скрапиш |
| 500 | Server Error — помилка сервера |


In [ ]:
# === HTTP GET REQUEST до rbc.ua ===

# User-Agent — обов'язковий для ввічливого scraping
# Ідентифікує нашого бота та ціль запиту
HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "uk,en;q=0.9",   # просимо українську мову
    "Accept": "text/html,application/xhtml+xml",
}

RBC_URL = "https://www.rbc.ua/"

# Виконуємо GET запит
# timeout=10 — Python чекатиме максимум 10 секунд
# якщо сервер не відповідає — кидає requests.exceptions.Timeout
print(f"Надсилаємо GET запит до {RBC_URL} ...")
response = requests.get(RBC_URL, headers=HEADERS, timeout=10)

# ── Аналіз відповіді ──────────────────────────────────────────────
print(f"\nStatus Code:   {response.status_code}")   # 200 = OK
print(f"Content-Type:  {response.headers.get('Content-Type', '—')}")
print(f"Розмір HTML:   {len(response.text):,} символів")
print(f"Кодування:     {response.encoding}")

print("\n=== Важливі HTTP Response Headers ===")
important_headers = ["Server", "Content-Type", "Content-Length",
                     "Cache-Control", "X-Frame-Options", "Strict-Transport-Security"]
for h in important_headers:
    val = response.headers.get(h)
    if val:
        print(f"  {h}: {val}")

print("\n=== Перші 500 символів HTML ===")
print(response.text[:500])

# 4. Аналіз HTML структури

Перш ніж писати парсер — потрібно **зрозуміти HTML структуру сайту**.

## Як досліджувати HTML?

**Спосіб 1 — DevTools браузера:**
1. Відкрий https://www.rbc.ua/ у браузері
2. Натисни `F12` → вкладка **Elements**
3. Натисни на елемент новини → бачиш HTML
4. Копіюй CSS class / тег

**Спосіб 2 — Python (BeautifulSoup):**
```python
# Переглянути відформатований HTML
print(soup.prettify()[:2000])

# Знайти всі унікальні теги
tags = {tag.name for tag in soup.find_all()}

# Знайти всі унікальні класи певного тегу
classes = {div.get('class', [''])[0] for div in soup.find_all('div')}
```

## Що ми шукаємо на rbc.ua?

Кожна новина (article) містить:
- **title** — заголовок
- **url** — посилання
- **category** — розділ (Економіка, Суспільство, Спорт...)
- **description** — короткий опис (якщо є)
- **date/time** — дата публікації


In [ ]:
# === ДОСЛІДЖЕННЯ HTML СТРУКТУРИ ===

# Парсимо вже завантажений HTML
soup = BeautifulSoup(response.text, "html.parser")

# ── Крок 1: загальна структура ────────────────────────────────────
print("=== Заголовок сторінки ===")
title_tag = soup.find("title")
print(f"  <title>: {title_tag.text if title_tag else 'не знайдено'}")

# ── Крок 2: всі посилання на новини ──────────────────────────────
# Шукаємо посилання які ведуть на статті rbc.ua
all_links = soup.find_all("a", href=True)
news_links = [
    a for a in all_links
    if a.get("href", "").startswith("/ukr/news/")
    or "/news/" in a.get("href", "")
]
print(f"\nПосилань на новини: {len(news_links)}")
for link in news_links[:5]:
    print(f"  {link.get('href', '')[:60]:60s} → {link.text.strip()[:50]}")

# ── Крок 3: шукаємо контейнери новин ─────────────────────────────
print("\n=== Типові контейнери новин ===")

# Стратегія: шукаємо div/article з class що містить 'news' або 'item'
# Різні сайти використовують різні назви класів
candidates = [
    ("article", {}),
    ("div", {"class": "newsline__item"}),
    ("div", {"class": "news-item"}),
    ("div", {"class": "news-feed__item"}),
    ("li", {"class": "newsline__item"}),
]
for tag, attrs in candidates:
    found = soup.find_all(tag, attrs) if attrs else soup.find_all(tag)
    if found:
        print(f"  <{tag} {list(attrs.values())}> → знайдено: {len(found)} елементів")

# ── Крок 4: prettify перших 1000 символів ────────────────────────
print("\n=== Перші 1000 символів відформатованого HTML ===")
print(soup.prettify()[:1000])

# 5. BeautifulSoup: Парсинг HTML

`BeautifulSoup` перетворює HTML-рядок на **Python-об'єкт** з яким можна взаємодіяти.

## Основні методи

| Метод | Що робить | Повертає |
|-------|-----------|----------|
| `soup.find("div")` | Перший `<div>` | Tag або None |
| `soup.find("div", class_="item")` | Перший `<div class="item">` | Tag або None |
| `soup.find_all("a")` | Всі `<a>` теги | list[Tag] |
| `soup.select(".item")` | CSS-selector | list[Tag] |
| `soup.select_one(".item a")` | Перший CSS-match | Tag або None |
| `tag.text` | Текст усередині | str |
| `tag.get_text(strip=True)` | Текст без зайвих пробілів | str |
| `tag["href"]` | Значення атрибуту | str |
| `tag.get("href")` | Безпечно (None замість KeyError) | str або None |

## DOM-дерево

```
<html>
 └── <body>
      └── <div class="newsline">
           ├── <div class="newsline__item">  ← один блок новини
           │    ├── <a href="/ukr/news/...">Заголовок</a>
           │    ├── <span class="category">Економіка</span>
           │    └── <time>2024-05-08</time>
           └── <div class="newsline__item">
                └── ...
```


In [ ]:
# === BeautifulSoup: БАЗОВІ ОПЕРАЦІЇ (демо на прикладі) ===

# Простий HTML що імітує структуру новинного сайту
sample_html = '''
<html><body>
  <div class="newsline">
    <div class="newsline__item">
      <a class="newsline__item-title" href="/ukr/news/2024/1234/">
        Уряд затвердив новий бюджет на 2024 рік
      </a>
      <span class="newsline__item-category">Економіка</span>
      <time class="newsline__item-date" datetime="2024-05-08T10:30:00">
        08 травня 2024, 10:30
      </time>
    </div>
    <div class="newsline__item">
      <a class="newsline__item-title" href="/ukr/news/2024/5678/">
        Збірна України перемогла у фіналі
      </a>
      <span class="newsline__item-category">Спорт</span>
      <time class="newsline__item-date" datetime="2024-05-08T09:15:00">
        08 травня 2024, 09:15
      </time>
    </div>
  </div>
</body></html>
'''

sample_soup = BeautifulSoup(sample_html, "html.parser")

print("=== find() — перший елемент ===")
first_item = sample_soup.find("div", class_="newsline__item")
print(f"  Тип: {type(first_item).__name__}")
print()

print("=== .text та .get_text() ===")
title_a = first_item.find("a")
print(f"  .text:                {title_a.text!r}")
print(f"  .get_text(strip=True): {title_a.get_text(strip=True)!r}")
print()

print("=== Атрибути ===")
print(f"  title_a['href']:     {title_a['href']!r}")
print(f"  title_a.get('href'): {title_a.get('href')!r}")
print()

print("=== find_all() — всі елементи ===")
all_items = sample_soup.find_all("div", class_="newsline__item")
print(f"  Знайдено новин: {len(all_items)}")
for item in all_items:
    title = item.find("a").get_text(strip=True)
    cat   = item.find("span").get_text(strip=True)
    date  = item.find("time").get("datetime", "")
    print(f"  [{cat:10s}] {title[:50]} ({date[:10]})")

print()
print("=== CSS Selectors: select() та select_one() ===")
# CSS selector: елемент.клас→дочірній елемент
titles_css = sample_soup.select("div.newsline__item a")
print(f"  select('div.newsline__item a'): {len(titles_css)} елементів")
for t in titles_css:
    print(f"  → {t.get_text(strip=True)[:50]}")

# 6. Парсинг RBC.ua: Функція витягування новин

Тепер пишемо **реальну функцію** для парсингу rbc.ua.

## Стратегія пошуку новин

RBC Ukraine використовує кілька форматів розміщення новин на головній сторінці:

1. **Блоки новинної стрічки** — список статей з заголовком, категорією, датою
2. **Featured блоки** — топові новини зверху

Наш парсер використовує **каскадний підхід**: пробуємо різні селектори,
повертаємо перший результат що дав дані.

```
Спробуй selector_1
  → якщо знайшов елементи → парси
  → якщо ні → спробуй selector_2
  → ...
```

Це важливо тому що **сайти змінюють HTML** з часом.


In [ ]:
# === ФУНКЦІЯ ПАРСИНГУ RBC.UA (виправлена) ===


BASE_URL = "https://www.rbc.ua"

def parse_rbc_news(html: str) -> list[dict]:
    """
    Парсить HTML сторінки rbc.ua → повертає список новин.
    Кожна новина: {title, url, category, description, datetime}
    """
    soup = BeautifulSoup(html, "html.parser")
    news_list = []

    # ── Стратегія 1: специфічні CSS-класи контейнерів ─────────────
    # НЕ включаємо ("article", None) — він знаходить ВСІ article-теги
    # (шапку, банери, блоки реклами) і викликає ранній break.
    item_selectors = [
        ("div", "newsline__item"),
        ("li",  "newsline__item"),
        ("div", "news-feed__item"),
        ("div", "news-item"),
        ("article", "news-item"),
        ("article", "article-item"),
    ]

    items = []
    for tag, cls in item_selectors:
        found = soup.find_all(tag, class_=cls)
        if found:
            items = found
            break

    # ── Парсинг знайдених контейнерів ────────────────────────────
    seen_urls = set()

    for item in items:
        title_tag = (
            item.find("a", class_=lambda c: c and "title" in c)
            or item.find("h2")
            or item.find("h3")
            or item.find("a")
        )
        if not title_tag:
            continue

        title = title_tag.get_text(strip=True)
        if len(title) < 10:
            continue

        href = title_tag.get("href", "") or (
            item.find("a").get("href", "") if item.find("a") else ""
        )
        full_url = href if href.startswith("http") else BASE_URL + href

        if full_url in seen_urls:
            continue
        seen_urls.add(full_url)

        cat_tag = item.find(
            lambda t: t.name in ("span", "div", "a")
            and any("categor" in c or "rubric" in c or "section" in c
                    for c in (t.get("class") or []))
        )
        category = cat_tag.get_text(strip=True) if cat_tag else ""

        desc_tag = item.find("p") or item.find(
            lambda t: t.name == "div"
            and any(k in " ".join(t.get("class") or [])
                    for k in ("desc", "lead", "anons", "text"))
        )
        description = desc_tag.get_text(strip=True) if desc_tag else ""
        if description == title:
            description = ""

        time_tag = item.find("time")
        if time_tag:
            dt = time_tag.get("datetime", "") or time_tag.get_text(strip=True)
        else:
            dt_tag = item.find(lambda t: any(
                k in " ".join(t.get("class") or []) for k in ("date", "time", "ago")
            ))
            dt = dt_tag.get_text(strip=True) if dt_tag else ""

        news_list.append({
            "title":       title,
            "url":         full_url,
            "category":    category,
            "description": description,
            "datetime":    dt[:19],
        })

    # ── Стратегія 2: пряме сканування посилань ────────────────────
    # Запускається якщо: контейнерів не знайдено АБО знайдено, але
    # парсинг дав 0 результатів (БАГ 3 — виправлено тут).
    if not news_list:
        all_links = soup.find_all("a", href=True)
        for a in all_links:
            href = a.get("href", "")
            if "/news/" not in href:
                continue

            # Текст посилання може містити "21:24\nЗаголовок новини"
            # — розбиваємо і беремо найдовшу частину як заголовок
            raw_text = a.get_text(separator=" ", strip=True)
            # Видаляємо часові мітки (HH:MM) з початку тексту
            title = re.sub(r"^\d{1,2}:\d{2}\s*", "", raw_text).strip()
            if len(title) < 10:
                continue

            full_url = href if href.startswith("http") else BASE_URL + href
            if full_url in seen_urls:
                continue
            seen_urls.add(full_url)

            # Час публікації — витягуємо з тексту якщо є "HH:MM" на початку
            time_match = re.match(r"^(\d{1,2}:\d{2})", raw_text)
            dt = time_match.group(1) if time_match else ""

            news_list.append({
                "title":       title,
                "url":         full_url,
                "category":    "",
                "description": "",
                "datetime":    dt,
            })

    return news_list


# ── Тестуємо ──────────────────────────────────────────────────────
rbc_news = parse_rbc_news(response.text)

print(f"Знайдено новин: {len(rbc_news)}\n")
for item in rbc_news[:10]:
    cat = f"[{item['category']}]" if item['category'] else ""
    dt  = item['datetime'] if item['datetime'] else "дата невідома"
    print(f"  {dt:5s}  {item['title'][:65]}")

In [ ]:
# === ЗБИРАЄМО НОВИНИ З КІЛЬКОХ СТОРІНОК ===
# rbc.ua має розділи з окремими URL, можна збирати з декількох

# Список URL для збору новин (головна + кілька розділів)
PAGES_TO_SCRAPE = [
    "https://www.rbc.ua/",
    "https://www.rbc.ua/ukr/news/",
]
DELAY = 1.5   # ввічлива пауза між запитами (секунди)

all_news = []
seen_urls = set()

for page_url in PAGES_TO_SCRAPE:
    print(f"Завантажуємо: {page_url} ...", end=" ")

    try:
        r = requests.get(page_url, headers=HEADERS, timeout=15)

        if r.status_code != 200:
            print(f"HTTP {r.status_code} — пропускаємо")
            continue

        page_news = parse_rbc_news(r.text)

        # Додаємо тільки нові (не дублікати)
        new_count = 0
        for item in page_news:
            if item["url"] not in seen_urls and item["title"]:
                seen_urls.add(item["url"])
                all_news.append(item)
                new_count += 1

        print(f"HTTP {r.status_code} → +{new_count} нових (всього: {len(all_news)})")

    except requests.exceptions.RequestException as e:
        print(f"Помилка: {e}")

    time.sleep(DELAY)   # ввічлива пауза — не перевантажуємо сервер!

print(f"\nРезультат: {len(all_news)} унікальних новин")

# 7. pandas DataFrame: Структуруємо дані

Тепер маємо `list[dict]` — це рівно те що потрібно для `pd.DataFrame()`.

```python
# list[dict] → DataFrame — стандартний патерн у Data Science
df = pd.DataFrame(news_list)
```

- Кожен `dict` → **рядок** таблиці
- Ключі `dict` → **назви колонок**


In [ ]:
# === КОНВЕРТУЄМО У pandas DataFrame ===

df = pd.DataFrame(all_news)

print(f"Форма DataFrame: {df.shape}  (рядки × колонки)")
print(f"Колонки: {list(df.columns)}")
print()

# Якщо зібрали 0 новин — використовуємо fallback тестові дані
if df.empty or len(df) < 3:
    print("Мало новин зібрано (можливо rbc.ua використовує JavaScript).")
    print("Використовуємо тестовий набір даних для демонстрації pipeline.\n")

    sample_news = [
        {"title": "НБУ підвищив облікову ставку до 15% річних",
         "url": "https://www.rbc.ua/ukr/news/nbu-1",
         "category": "Економіка", "description": "Рішення набуде чинності з наступного тижня.",
         "datetime": "2024-05-08T10:30:00"},
        {"title": "Уряд схвалив пакет допомоги для малого бізнесу",
         "url": "https://www.rbc.ua/ukr/news/nbu-2",
         "category": "Економіка", "description": "Загальний обсяг фінансування — 5 млрд грн.",
         "datetime": "2024-05-08T09:15:00"},
        {"title": "ЗСУ відбили атаку у Харківській області",
         "url": "https://www.rbc.ua/ukr/news/nbu-3",
         "category": "Суспільство", "description": "Ворог зазнав значних втрат.",
         "datetime": "2024-05-08T08:45:00"},
        {"title": "Збірна України з футболу готується до відбору на ЧС",
         "url": "https://www.rbc.ua/ukr/news/nbu-4",
         "category": "Спорт", "description": "Тренування розпочнуться цього тижня.",
         "datetime": "2024-05-08T08:00:00"},
        {"title": "Ринок нерухомості Києва: ціни зросли на 12%",
         "url": "https://www.rbc.ua/ukr/news/nbu-5",
         "category": "Нерухомість", "description": "Попит на квартири продовжує зростати.",
         "datetime": "2024-05-07T17:30:00"},
        {"title": "МВФ схвалив черговий транш для України на 880 млн доларів",
         "url": "https://www.rbc.ua/ukr/news/nbu-6",
         "category": "Економіка", "description": "Кошти підуть на підтримку бюджету.",
         "datetime": "2024-05-07T16:00:00"},
        {"title": "Уряд планує реформу освіти у 2025 році",
         "url": "https://www.rbc.ua/ukr/news/edu-1",
         "category": "Суспільство", "description": "Буде впроваджено нові стандарти навчання.",
         "datetime": "2024-05-07T15:00:00"},
        {"title": "Енергосистема України витримала нічні удари",
         "url": "https://www.rbc.ua/ukr/news/energy-1",
         "category": "Суспільство", "description": "Світло є у всіх регіонах.",
         "datetime": "2024-05-07T14:00:00"},
        {"title": "Нові санкції ЄС проти Росії: що потрапило до списку",
         "url": "https://www.rbc.ua/ukr/news/sanctions-1",
         "category": "Світ", "description": "14-й пакет санкцій включає понад 100 компаній.",
         "datetime": "2024-05-07T12:00:00"},
        {"title": "Туристичний сезон в Україні: де відпочити влітку",
         "url": "https://www.rbc.ua/ukr/news/tourism-1",
         "category": "Суспільство", "description": "Популярні напрямки та ціни.",
         "datetime": "2024-05-07T11:00:00"},
        {"title": "Інфляція в Україні уповільнилась до 3.2%",
         "url": "https://www.rbc.ua/ukr/news/inflation-1",
         "category": "Економіка", "description": "Найнижчий показник за останні роки.",
         "datetime": "2024-05-06T10:00:00"},
        {"title": "Рада ухвалила закон про мобілізацію у другому читанні",
         "url": "https://www.rbc.ua/ukr/news/mob-1",
         "category": "Суспільство", "description": "Закон набере чинності через 30 днів.",
         "datetime": "2024-05-06T09:00:00"},
        {"title": "Гривня зміцнилась до 38.5 за долар",
         "url": "https://www.rbc.ua/ukr/news/hryvnia-1",
         "category": "Економіка", "description": "НБУ проводить інтервенції на валютному ринку.",
         "datetime": "2024-05-06T08:00:00"},
        {"title": "Кличко оголосив нові проекти розвитку Києва",
         "url": "https://www.rbc.ua/ukr/news/kyiv-1",
         "category": "Суспільство", "description": "Реконструкція парків та доріг.",
         "datetime": "2024-05-05T17:00:00"},
        {"title": "Чемпіонат України з боксу: підсумки фіналів",
         "url": "https://www.rbc.ua/ukr/news/boxing-1",
         "category": "Спорт", "description": "Золото завоювали спортсмени з Харкова та Одеси.",
         "datetime": "2024-05-05T16:00:00"},
    ]
    df = pd.DataFrame(sample_news)
    print(f"Тестовий набір: {df.shape}")

# Переглянемо перші рядки
df.head(5)

In [ ]:
# === ДОСЛІДЖЕННЯ DataFrame ===

print("=== 1. Структура ===")
print(f"  Рядків: {df.shape[0]}, Колонок: {df.shape[1]}")
print(f"  Колонки: {list(df.columns)}")
print()

print("=== 2. Розподіл по категоріях ===")
cat_counts = df["category"].value_counts()
if not cat_counts.empty and cat_counts.iloc[0] != "":
    print(cat_counts.to_string())
else:
    print("  (категорії порожні — rbc.ua може ховати їх у JS)")
print()

print("=== 3. Сортування — найновіші новини ===")
df_sorted = df[df["datetime"] != ""].sort_values("datetime", ascending=False)
print(df_sorted[["datetime", "category", "title"]].head(5).to_string(index=False))
print()

print("=== 4. Фільтрування — новини категорії 'Економіка' ===")
eco_df = df[df["category"] == "Економіка"]
print(f"  Знайдено: {len(eco_df)} новин")
if not eco_df.empty:
    print(eco_df[["title", "datetime"]].to_string(index=False))
print()

print("=== 5. Довжина заголовків ===")
df["title_len"] = df["title"].str.len()
print(df["title_len"].describe())

# 8. Text Mining: Аналіз заголовків

**Text Mining** — витягування корисної інформації з неструктурованого тексту.

Text Mining будемо робити по **заголовках новин** (колонка `title`).

## Pipeline:

```
raw titles ("Уряд затвердив бюджет...")
   ↓  lower()
lowercase
   ↓  re.sub(r'[^а-яёіїєа-яa-z ]', '')
тільки літери + пробіли
   ↓  .split()
["уряд", "затвердив", "бюджет", ...]
   ↓  видалити українські стоп-слова
["уряд", "бюджет", "затвердив", ...]
   ↓  Counter()
{"уряд": 5, "бюджет": 4, ...}
   ↓
WordCloud + BarChart
```

## Українські Stopwords

**Stopwords** — слова що зустрічаються часто, але не несуть змісту:
`та`, `в`, `і`, `на`, `що`, `до`, `для`, `як`, `за`, `але`...

Без видалення стоп-слів — вони домінують і приховують справжні теми.


In [ ]:
# === УКРАЇНСЬКІ СТОП-СЛОВА ===

# Стоп-слова — слова без семантичного навантаження в новинних текстах
STOPWORDS_UK = {
    # Службові слова
    "та", "і", "й", "а", "але", "або", "чи", "що", "як", "бо", "якщо",
    "то", "вже", "ще", "не", "ні", "так", "ось", "от", "ж", "б", "би",
    # Прийменники
    "в", "у", "на", "з", "із", "зі", "до", "від", "для", "за", "по",
    "під", "над", "між", "через", "про", "при", "без", "після",
    "перед", "під", "о", "об", "серед",
    # Займенники
    "він", "вона", "воно", "вони", "ми", "ви", "я", "ти", "це",
    "той", "та", "ті", "цей", "ця", "ці",
    # Артикли/частки
    "де", "коли", "хто", "яка", "який", "яке", "які",
    # Дієслова-зв'язки
    "є", "був", "була", "були", "буде", "буде", "стало", "стала",
    # Числівники-слова
    "один", "два", "три", "тисяч", "мільйон",
    # Короткі слова
    "мав", "має", "мають", "має", "цьому", "цього",
    # Англійські (можуть зустрічатися)
    "the", "a", "an", "and", "or", "in", "of", "to", "is", "for",
    # Загальні слова новин (без конкретного змісту)
    "повідомляє", "заявив", "сказав", "сказала",
}

print(f"Стоп-слів у списку: {len(STOPWORDS_UK)}")
print("Приклади:", ", ".join(sorted(STOPWORDS_UK)[:20]))

In [ ]:
# === ТОКЕНІЗАЦІЯ ТА ОЧИЩЕННЯ ТЕКСТУ ===

# Об'єднуємо ВСІ заголовки в один текст
all_titles = " ".join(df["title"].tolist())

print(f"Загальний текст заголовків: {len(all_titles):,} символів")
print(f"Перші 300 символів:")
print(all_titles[:300])
print()

# ── Крок 1: lowercase ─────────────────────────────────────────────
text_lower = all_titles.lower()

# ── Крок 2: видалення пунктуації та цифр ──────────────────────────
# Залишаємо тільки кириличні/латинські літери та пробіли
# [^а-яёіїєa-z\s] — видаляємо все що НЕ є літерою або пробілом
text_clean = re.sub(r"[^а-яёіїєa-zа-я\s]", " ", text_lower)
text_clean = re.sub(r"\s+", " ", text_clean).strip()  # прибираємо зайві пробіли

# ── Крок 3: токенізація — розбиваємо на слова ──────────────────────
tokens = text_clean.split()

print(f"Токенів після tokenize: {len(tokens):,}")
print(f"Перші 20 токенів: {tokens[:20]}")
print()

# ── Крок 4: видалення стоп-слів + короткі слова ────────────────────
filtered_tokens = [
    word for word in tokens
    if word not in STOPWORDS_UK
    and len(word) >= 3   # ігноруємо слова коротші за 3 символи
]

print(f"Токенів до фільтрації:    {len(tokens):,}")
print(f"Токенів після фільтрації: {len(filtered_tokens):,}")
print(f"Видалено стоп-слів:       {len(tokens) - len(filtered_tokens):,}")
print()

# ── Крок 5: підрахунок частот ──────────────────────────────────────
word_freq = Counter(filtered_tokens)

print("=== Топ-20 слів у заголовках RBC Ukraine ===")
for rank, (word, count) in enumerate(word_freq.most_common(20), 1):
    bar = "█" * count
    print(f"  {rank:2d}. {word:25s} {count:3d}  {bar}")

# 9. Візуалізація: Bar Chart частотності


In [ ]:
# === MATPLOTLIB: HORIZONTAL BAR CHART ===

top_n = 15
top_words = word_freq.most_common(top_n)
words  = [w for w, c in top_words]
counts = [c for w, c in top_words]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# ── Графік 1: горизонтальний bar chart ───────────────────────────
colors = plt.cm.Blues([0.4 + 0.6 * i / top_n for i in range(top_n)])
bars = axes[0].barh(
    words[::-1], counts[::-1],
    color=colors[::-1], edgecolor="white", linewidth=0.5
)

# Підписи значень на барах
for bar, count in zip(bars, counts[::-1]):
    axes[0].text(
        bar.get_width() + 0.05, bar.get_y() + bar.get_height() / 2,
        str(count), va="center", fontsize=10
    )

axes[0].set_title(
    f"Топ-{top_n} слів у заголовках RBC Ukraine",
    fontsize=13, fontweight="bold", pad=12
)
axes[0].set_xlabel("Частота вживання", fontsize=11)
axes[0].set_xlim(0, max(counts) * 1.15)
axes[0].spines["top"].set_visible(False)
axes[0].spines["right"].set_visible(False)

# ── Графік 2: розподіл по категоріях ────────────────────────────
cat_counts = df["category"].value_counts()
if not cat_counts.empty and cat_counts.index[0] != "":
    cat_colors = plt.cm.Set3([i / len(cat_counts) for i in range(len(cat_counts))])
    axes[1].barh(
        cat_counts.index.tolist()[::-1],
        cat_counts.values.tolist()[::-1],
        color=cat_colors[::-1]
    )
    axes[1].set_title("Кількість новин по категоріях", fontsize=13, fontweight="bold", pad=12)
    axes[1].set_xlabel("Кількість новин", fontsize=11)
    axes[1].spines["top"].set_visible(False)
    axes[1].spines["right"].set_visible(False)
else:
    axes[1].text(0.5, 0.5, "Категорії не знайдено\n(сайт використовує JS)",
                 ha="center", va="center", transform=axes[1].transAxes, fontsize=12)
    axes[1].axis("off")

plt.suptitle("Text Mining: RBC Ukraine — Аналіз новин",
             fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# 10. WordCloud з підтримкою кирилиці

`WordCloud` — візуалізація де **розмір слова пропорційний частоті**.

## Важливо для кирилиці!

За замовчуванням `WordCloud` може не відображати українські символи.
Потрібно вказати **шрифт з підтримкою кирилиці**.

На Linux/WSL підходять шрифти:
- `/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf`
- `/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf`

На Windows:
- `C:/Windows/Fonts/arial.ttf`
- `C:/Windows/Fonts/times.ttf`


In [ ]:
# === WORDCLOUD З ПІДТРИМКОЮ КИРИЛИЦІ ===

import os

# Пошук шрифту що підтримує кирилицю
FONT_CANDIDATES = [
    # Linux / WSL
    "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
    "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",
    "/usr/share/fonts/truetype/ubuntu/Ubuntu-R.ttf",
    # Windows
    "C:/Windows/Fonts/arial.ttf",
    "C:/Windows/Fonts/times.ttf",
    "C:/Windows/Fonts/calibri.ttf",
    # macOS
    "/System/Library/Fonts/Helvetica.ttc",
    "/Library/Fonts/Arial Unicode.ttf",
]

font_path = None
for fp in FONT_CANDIDATES:
    if os.path.exists(fp):
        font_path = fp
        break

if font_path:
    print(f"Знайдено шрифт: {font_path}")
else:
    print("Шрифт не знайдено — WordCloud може не відображати кирилицю")
    print("Встанови: sudo apt-get install fonts-dejavu-core  (Linux/WSL)")

In [ ]:
# === ГЕНЕРУЄМО WORDCLOUD ===

try:
    from wordcloud import WordCloud

    # WordCloud з частот (точніше ніж з сирого тексту)
    wc_kwargs = {
        "width": 900,
        "height": 500,
        "background_color": "white",
        "colormap": "Blues",
        "max_words": 80,
        "min_font_size": 10,
        "max_font_size": 110,
        "prefer_horizontal": 0.8,
    }
    if font_path:
        wc_kwargs["font_path"] = font_path

    wc = WordCloud(**wc_kwargs).generate_from_frequencies(dict(word_freq))

    # ── Головний WordCloud + топ-15 рядом ────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    axes[0].imshow(wc, interpolation="bilinear")
    axes[0].axis("off")
    axes[0].set_title(
        "WordCloud — заголовки RBC Ukraine\n(розмір слова = частота)",
        fontsize=13, fontweight="bold", pad=10
    )

    # Топ-15 слів поряд для порівняння
    top15 = word_freq.most_common(15)
    colors_bar = plt.cm.Blues([0.4 + 0.6 * i / 15 for i in range(15)])
    axes[1].barh(
        [w for w, c in top15][::-1],
        [c for w, c in top15][::-1],
        color=colors_bar
    )
    axes[1].set_title("Топ-15 слів (частоти)", fontsize=13, fontweight="bold", pad=10)
    axes[1].set_xlabel("Частота")
    axes[1].spines["top"].set_visible(False)
    axes[1].spines["right"].set_visible(False)

    plt.suptitle("RBC Ukraine — Text Mining Analysis",
                 fontsize=15, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()

    # ── Dark theme WordCloud ──────────────────────────────────────
    wc_dark_kwargs = {
        "width": 900, "height": 450,
        "background_color": "#1a1a2e",
        "colormap": "plasma",
        "max_words": 60,
    }
    if font_path:
        wc_dark_kwargs["font_path"] = font_path

    wc_dark = WordCloud(**wc_dark_kwargs).generate_from_frequencies(dict(word_freq))

    plt.figure(figsize=(13, 5))
    plt.imshow(wc_dark, interpolation="bilinear")
    plt.axis("off")
    plt.title("WordCloud (dark theme) — RBC Ukraine Headlines",
              fontsize=13, fontweight="bold", color="white", pad=10)
    plt.gca().set_facecolor("#1a1a2e")
    plt.gcf().set_facecolor("#1a1a2e")
    plt.tight_layout()
    plt.show()

except ImportError:
    print("wordcloud не встановлено. Виконай: !pip install wordcloud")

# 11. Додаткова візуалізація


In [ ]:
# === КОМПЛЕКСНА DASHBOARD-ВІЗУАЛІЗАЦІЯ ===

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("RBC Ukraine — News Analytics Dashboard",
             fontsize=16, fontweight="bold", y=1.01)

# ── Графік 1 (top-left): Топ-10 слів ────────────────────────────
top10 = word_freq.most_common(10)
w10, c10 = [w for w, c in top10], [c for w, c in top10]

bars1 = axes[0, 0].barh(w10[::-1], c10[::-1],
                         color="#2196F3", edgecolor="white")
for bar, cnt in zip(bars1, c10[::-1]):
    axes[0, 0].text(bar.get_width() + 0.05,
                    bar.get_y() + bar.get_height() / 2,
                    str(cnt), va="center", fontsize=9)
axes[0, 0].set_title("Топ-10 слів у заголовках", fontweight="bold")
axes[0, 0].set_xlabel("Частота")
axes[0, 0].spines["top"].set_visible(False)
axes[0, 0].spines["right"].set_visible(False)

# ── Графік 2 (top-right): Розподіл по категоріях (pie) ──────────
cat_data = df["category"].value_counts()
cat_data = cat_data[cat_data.index != ""]  # прибираємо порожні

if not cat_data.empty:
    axes[0, 1].pie(
        cat_data.values,
        labels=cat_data.index,
        autopct="%1.0f%%",
        colors=plt.cm.Set3([i / len(cat_data) for i in range(len(cat_data))]),
        startangle=90, pctdistance=0.85
    )
    axes[0, 1].set_title("Розподіл по категоріях", fontweight="bold")
else:
    axes[0, 1].text(0.5, 0.5, "Категорії\nне знайдено",
                    ha="center", va="center", fontsize=12,
                    transform=axes[0, 1].transAxes)
    axes[0, 1].axis("off")

# ── Графік 3 (bottom-left): Довжина заголовків ───────────────────
title_lengths = df["title"].str.len()
axes[1, 0].hist(title_lengths, bins=20, color="#4CAF50",
                edgecolor="white", linewidth=0.5)
axes[1, 0].axvline(title_lengths.mean(), color="red",
                   linestyle="--", linewidth=1.5,
                   label=f"Середня: {title_lengths.mean():.0f} символів")
axes[1, 0].set_title("Розподіл довжини заголовків", fontweight="bold")
axes[1, 0].set_xlabel("Символів у заголовку")
axes[1, 0].set_ylabel("Кількість новин")
axes[1, 0].legend(fontsize=9)
axes[1, 0].spines["top"].set_visible(False)
axes[1, 0].spines["right"].set_visible(False)

# ── Графік 4 (bottom-right): Word frequency — лог-шкала ─────────
top20 = word_freq.most_common(20)
ranks = list(range(1, len(top20) + 1))
freqs = [c for _, c in top20]

axes[1, 1].plot(ranks, freqs, "o-", color="#FF5722",
                linewidth=2, markersize=6)
axes[1, 1].set_yscale("log")
axes[1, 1].set_title("Zipf's Law: ранг vs частота (лог-шкала)",
                     fontweight="bold")
axes[1, 1].set_xlabel("Ранг слова")
axes[1, 1].set_ylabel("Частота (log)")
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].spines["top"].set_visible(False)
axes[1, 1].spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

# 12. Export: Зберігаємо дані


In [ ]:
# === EXPORT: CSV + JSON ===
import json

# ── CSV — для Excel / pandas / SQL import ─────────────────────────
csv_path = "rbc_news.csv"
df.to_csv(csv_path, index=False, encoding="utf-8-sig")  # utf-8-sig для коректного Excel
print(f"CSV збережено: {csv_path}  ({os.path.getsize(csv_path):,} bytes)")

# Перевірка
df_check = pd.read_csv(csv_path)
print(f"   Перевірка: {len(df_check)} рядків")
print(df_check.head(3).to_string(index=False))
print()

# ── JSON — для JavaScript / API / MongoDB ─────────────────────────
json_path = "rbc_news.json"
records = df.drop(columns=["title_len"], errors="ignore").to_dict("records")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)
print(f"JSON збережено: {json_path}  ({os.path.getsize(json_path):,} bytes)")

# Перший запис
with open(json_path, encoding="utf-8") as f:
    loaded = json.load(f)
print("\nПерший запис JSON:")
print(json.dumps(loaded[0], ensure_ascii=False, indent=4))

# 13. Production Pattern: Robust HTTP GET

У реальних проектах HTTP запити можуть падати.
Ось патерн **з retry та timeout** для надійного scraping:


In [ ]:
# === PRODUCTION PATTERN: retry + exponential backoff ===

def robust_get(
    url: str,
    max_retries: int = 3,
    base_delay: float = 1.0
) -> requests.Response | None:
    """
    HTTP GET з timeout, retry та exponential backoff.
    Повертає Response або None якщо всі спроби вичерпано.
    """
    headers = {"User-Agent": "Mozilla/5.0"}

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(
                url,
                headers=headers,
                timeout=(5.0, 30.0),   # (connect timeout, read timeout)
            )

            # Rate limit — сервер каже "занадто часто"
            if response.status_code == 429:
                wait = int(response.headers.get("Retry-After", base_delay * attempt))
                print(f"  HTTP 429 — чекаємо {wait}s...")
                time.sleep(wait)
                continue

            response.raise_for_status()  # кидає HTTPError якщо 4xx/5xx
            return response

        except requests.exceptions.ConnectionError:
            delay = base_delay * (2 ** (attempt - 1))  # 1s, 2s, 4s
            print(f"  Спроба {attempt}/{max_retries}: ConnectionError — чекаємо {delay:.0f}s")
            time.sleep(delay)

        except requests.exceptions.Timeout:
            print(f"  Спроба {attempt}/{max_retries}: Timeout")

        except requests.exceptions.HTTPError as e:
            print(f"  HTTP помилка: {e}")
            return None

    print(f"  Всі {max_retries} спроби вичерпано — {url}")
    return None


# Тест
print("=== Тест robust_get() ===")
r = robust_get("https://www.rbc.ua/")
if r:
    print(f"HTTP {r.status_code} | {len(r.text):,} bytes | OK")

# 14. Практичні Вправи

---

## Вправа 1 — Beginner: Аналіз конкретного розділу

**Завдання:** Завантаж окремий розділ rbc.ua (наприклад, економіка або спорт).
Спарси заголовки та виведи їх.

Підказка: Спробуй URL:
- `https://www.rbc.ua/ukr/economy/`
- `https://www.rbc.ua/ukr/sport/`


In [ ]:
# === ВПРАВА 1: Завантаж окремий розділ rbc.ua ===

# TODO: вибери URL розділу
# SECTION_URL = "https://www.rbc.ua/ukr/economy/"

# TODO: зроби requests.get() з headers та timeout

# TODO: спарси через parse_rbc_news()

# TODO: виведи перші 10 заголовків

# ---- твій код нижче ----


## Вправа 2 — Beginner: Знайди найдовший заголовок

**Завдання:** Використовуючи `df` з новинами:
1. Знайди новину з **найдовшим** заголовком
2. Знайди новину з **найкоротшим** заголовком
3. Виведи обидва заголовки та їх довжину


In [ ]:
# === ВПРАВА 2: Найдовший і найкоротший заголовок ===

# TODO: додай колонку title_len = довжина заголовку
# df["title_len"] = df["title"].str.len()

# TODO: знайди рядок з максимальним title_len (df.nlargest(1, "title_len"))

# TODO: знайди рядок з мінімальним title_len (df.nsmallest(1, "title_len"))

# TODO: виведи обидва заголовки

# ---- твій код нижче ----


## Вправа 3 — Intermediate: Свій WordCloud по категорії

**Завдання:** Зроби WordCloud тільки для новин категорії **"Економіка"**.

Кроки:
1. Відфільтруй `df` де `category == "Економіка"`
2. Об'єднай заголовки
3. Токенізуй та очисти (використай вже написаний pipeline)
4. Побудуй `WordCloud` з палітрою `"Greens"`


In [ ]:
# === ВПРАВА 3: WordCloud по категорії "Економіка" ===

# TODO: відфільтруй df для категорії Економіка
# eco_df = df[df["category"] == "Економіка"]

# TODO: об'єднай заголовки в один текст
# eco_text = " ".join(eco_df["title"].tolist())

# TODO: lowercase + re.sub + split + фільтрація стоп-слів

# TODO: Counter() + WordCloud(colormap="Greens")

# ---- твій код нижче ----


## Вправа 4 — Intermediate: Bigrams (пари слів)

**Завдання:** Замість окремих слів — знайди найпопулярніші **пари слів** (bigrams).

Bigrams: `["уряд", "затвердив", "бюджет"]` → `[("уряд", "затвердив"), ("затвердив", "бюджет")]`

```python
# Генерація bigrams
bigrams = [(tokens[i], tokens[i+1]) for i in range(len(tokens)-1)]
bigram_freq = Counter(bigrams)
```

Виведи топ-10 bigrams.


In [ ]:
# === ВПРАВА 4: Bigrams ===

# TODO: використай filtered_tokens з основного pipeline
# bigrams = [(filtered_tokens[i], filtered_tokens[i+1]) for i in range(len(filtered_tokens)-1)]

# TODO: Counter(bigrams).most_common(10)

# TODO: виведи у форматі: "слово1 слово2" → count

# ---- твій код нижче ----


## Вправа 5 — Advanced: Порівняй два розділи

**Завдання:** Зроби **порівняльний WordCloud** для двох категорій новин.

1. Відфільтруй df для категорії A та категорії B
2. Зроби text mining для кожної
3. Побудуй `plt.subplots(1, 2)` — WordCloud ліворуч, WordCloud праворуч
4. Заголовок кожного subplot = назва категорії


In [ ]:
# === ВПРАВА 5: Порівняльний WordCloud двох категорій ===

# TODO: вибери дві категорії з df["category"].value_counts().index
# CAT_A = "Економіка"
# CAT_B = "Суспільство"

# TODO: text mining pipeline для кожної категорії

# TODO: plt.subplots(1, 2) → два WordCloud поряд

# ---- твій код нижче ----


# 15. Summary: Ключові Висновки

---

## Web Scraping Pipeline

```
requests.get(url, headers=..., timeout=...)
       ↓  HTTP 200 + HTML
response.text  →  HTML рядок
       ↓
BeautifulSoup(html, "html.parser")
       ↓  DOM-дерево
soup.find_all("div", class_="newsline__item")
       ↓
[{"title": ..., "url": ..., "category": ...}]
       ↓
pd.DataFrame(news_list)
       ↓
Counter(tokens) → most_common(20)
       ↓
WordCloud + matplotlib
```

---

## Що ти вивчив

| Концепція | Команда | Пам'ятай |
|-----------|---------|----------|
| HTTP GET | `requests.get(url, timeout=...)` | **Завжди** `timeout`! |
| HTML parsing | `BeautifulSoup(html, "html.parser")` | "html.parser" вбудований |
| Перший елемент | `soup.find("div", class_="x")` | None якщо не знайдено |
| Всі елементи | `soup.find_all("div", class_="x")` | Порожній list якщо ні |
| CSS selector | `soup.select(".item a")` | Як у браузері DevTools |
| Текст | `tag.get_text(strip=True)` | `strip=True` прибирає пробіли |
| Атрибут | `tag.get("href")` | `.get()` безпечніший ніж `[]` |
| Укр. стоп-слова | `STOPWORDS_UK = {"та", "і", ...}` | Без них — шум у WordCloud |
| WordCloud кирилиця | `font_path=...` | Потрібен шрифт з кирилицею |
| Export | `df.to_csv(...)` | `utf-8-sig` для Excel |
| Ввічливість | `time.sleep(1.0)` | Не перевантажуй сервер! |

---

## JavaScript-сайти

Якщо `requests` + `BeautifulSoup` повертає порожній результат —
сайт використовує **JavaScript для рендерингу контенту** (SPA/React/Vue).

У такому випадку потрібно:
- **Playwright** або **Selenium** — браузер який виконує JS
- **Офіційний API** сайту (якщо є)
- **RSS-стрічка** — багато новинних сайтів мають `/rss` або `/feed`

```python
# rbc.ua RSS (якщо доступно)
import feedparser
feed = feedparser.parse("https://www.rbc.ua/rss/")
for entry in feed.entries[:5]:
    print(entry.title, entry.published)
```

---

## Наступні кроки

- **feedparser** — парсинг RSS-стрічок (простіше ніж HTML!)
- **Scrapy** — фреймворк для великих scraping проєктів
- **Playwright** — для JS-сайтів
- **NLTK / spaCy** — глибший NLP для української мови
- **pandas + SQLAlchemy** — зберігання в SQL базу даних
- **Sentiment Analysis** — аналіз тональності новин
